# 資料處理

## 讀nc4

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


# 限制範圍抽樣 in USA

## 100個格點

In [ ]:
# ===== 只看美國本土範圍，並從中抽樣 100 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([
    lon_all_180[idx_us_mainland],
    lat_all[idx_us_mainland]
])

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# ===== 抽樣 100 個格點 =====
N_SAMPLE_TARGET = 100
n_sample = min(N_SAMPLE_TARGET, len(idx_us_mainland))

np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(
    len(idx_us_mainland),
    size=n_sample,
    replace=False
)

# 原始 global index
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後資料
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([
    lon_all_180[sample_idx_global],
    lat_all[sample_idx_global]
])

lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")

### DLinear

In [ ]:
import numpy as np
import pandas as pd

from darts import TimeSeries
from darts.models import DLinearModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


print("\n===== DLinear hyperparameter search (train/test = 80/20) =====")

# ============================================================
# 0) Fixed settings (不進搜尋的固定項)
# ============================================================
USE_COVARIATES = False
USE_REVIN      = True
RANDOM_STATE   = 42

PL_TRAINER_KWARGS = {
    "enable_progress_bar": False,
    "logger": False,
    "enable_checkpointing": False,
}

# ============================================================
# 1) 建 target DataFrame：n_sample 個格點 × T 時間點（原始單位）
#    你需要先在外部準備好：
#    - y_sample: (n_sample, T)
#    - time_index: pandas.DatetimeIndex, length = T
#    - n_sample: int
# ============================================================
SAMPLE_SIZE = int(n_sample)
T = int(y_sample.shape[1])

ts_df = pd.DataFrame(
    y_sample.T,
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T)
print("ts_df shape:", ts_df.shape)

# ============================================================
# 2) month covariate（保留；USE_COVARIATES=False 時不會用到）
# ============================================================
month_values = time_index.month.astype("float32")
month_df = pd.DataFrame({"month": month_values}, index=time_index)
month_ts = TimeSeries.from_dataframe(month_df.astype("float32"))

# ============================================================
# 3) 切成 80 / 20（train / test）
# ============================================================
train_frac = 0.8
cut_train = int(T * train_frac)

idx = ts_df.index
split_time = idx[cut_train]

ts_all_raw = TimeSeries.from_dataframe(ts_df)
train_raw, test_raw = ts_all_raw.split_before(split_time)
print(f"Train len (raw): {len(train_raw)}, Test len: {len(test_raw)}")

month_train, month_test = month_ts.split_before(split_time)
assert len(month_train) == len(train_raw)
assert len(month_test) == len(test_raw)

# ============================================================
# 4) 用 train 統計量標準化（只用 train 的 mean/std）
# ============================================================
train_df = ts_df.iloc[:cut_train]
mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)

ts_df_scaled  = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, test_ts = ts_all_scaled.split_before(split_time)

T_train, T_test = len(train_ts), len(test_ts)
print("\n[Scaled lengths] T_train =", T_train, "T_test =", T_test)

test_true_raw = ts_df.to_numpy(dtype=np.float32)[cut_train:, :]

def inverse_scale(x: np.ndarray) -> np.ndarray:
    return x * std_vec.values + mean_vec.values

def _ts_to_2d(ts: TimeSeries) -> np.ndarray:
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr

# ============================================================
# 5) 要搜尋的參數集合（你只要改這段）
# ============================================================
SEARCH_SPACE = {
    "INPUT_CHUNK_LENGTH":  [24, 36, 48],
    "OUTPUT_CHUNK_LENGTH": [12,24],
    "KERNEL_SIZE":         [15, 25, 35],
    "N_EPOCHS":            [30, 50],
    "BATCH_SIZE":          [32, 64, 128],
    "LR":                  [1e-4, 2e-4, 3e-4],
    "WEIGHT_DECAY":        [0.0, 1e-5],
    "CONST_INIT":          [True, False],
}

# ============================================================
# 6) Grid search：只輸出最佳參數組合（以 TEST RMSE 最小為準）
# ============================================================
keys = list(SEARCH_SPACE.keys())

best = {
    "rmse_test": np.inf,
    "mse_test": None,
    "mae_test": None,
    "r2_test": None,
    "params": None,
}

total = 1
for k in keys:
    total *= len(SEARCH_SPACE[k])
print("\nTotal combos:", total)

combo_id = 0
for in_len in SEARCH_SPACE["INPUT_CHUNK_LENGTH"]:
    for out_len in SEARCH_SPACE["OUTPUT_CHUNK_LENGTH"]:
        for ksize in SEARCH_SPACE["KERNEL_SIZE"]:
            for n_epochs in SEARCH_SPACE["N_EPOCHS"]:
                for bs in SEARCH_SPACE["BATCH_SIZE"]:
                    for lr in SEARCH_SPACE["LR"]:
                        for wd in SEARCH_SPACE["WEIGHT_DECAY"]:
                            for const_init in SEARCH_SPACE["CONST_INIT"]:
                                combo_id += 1

                                model_kwargs = dict(
                                    input_chunk_length=in_len,
                                    output_chunk_length=out_len,
                                    kernel_size=ksize,
                                    n_epochs=n_epochs,
                                    random_state=RANDOM_STATE,
                                    use_reversible_instance_norm=USE_REVIN,
                                    batch_size=bs,
                                    optimizer_kwargs={"lr": float(lr), "weight_decay": float(wd)},
                                    const_init=bool(const_init),
                                    pl_trainer_kwargs=PL_TRAINER_KWARGS,
                                    log_tensorboard=False,
                                    save_checkpoints=False,
                                )

                                model = DLinearModel(**model_kwargs)

                                fit_kwargs = {"series": train_ts, "verbose": False}
                                pred_kwargs = {"n": T_test, "verbose": False, "show_warnings": False}

                                if USE_COVARIATES:
                                    fit_kwargs["past_covariates"] = month_train
                                    fit_kwargs["future_covariates"] = month_train
                                    pred_kwargs["past_covariates"] = month_ts
                                    pred_kwargs["future_covariates"] = month_ts

                                try:
                                    model.fit(**fit_kwargs)
                                    pred_test = model.predict(**pred_kwargs)
                                except Exception:
                                    continue

                                pred_test_scaled = _ts_to_2d(pred_test)
                                pred_test_raw = inverse_scale(pred_test_scaled)

                                y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
                                y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

                                mse_test  = mean_squared_error(y_test_true, y_test_pred)
                                mae_test  = mean_absolute_error(y_test_true, y_test_pred)
                                rmse_test = float(np.sqrt(mse_test))
                                r2_test   = r2_score(y_test_true, y_test_pred)

                                if rmse_test < best["rmse_test"]:
                                    best["rmse_test"] = rmse_test
                                    best["mse_test"]  = float(mse_test)
                                    best["mae_test"]  = float(mae_test)
                                    best["r2_test"]   = float(r2_test)
                                    best["params"] = {
                                        "INPUT_CHUNK_LENGTH": in_len,
                                        "OUTPUT_CHUNK_LENGTH": out_len,
                                        "KERNEL_SIZE": ksize,
                                        "N_EPOCHS": n_epochs,
                                        "BATCH_SIZE": bs,
                                        "LR": float(lr),
                                        "WEIGHT_DECAY": float(wd),
                                        "CONST_INIT": bool(const_init),
                                        "USE_REVIN": bool(USE_REVIN),
                                        "USE_COVARIATES": bool(USE_COVARIATES),
                                        "RANDOM_STATE": int(RANDOM_STATE),
                                    }

# ============================================================
# 7) 只輸出最佳參數組合（不存檔）
# ============================================================
print("\n===== BEST PARAMS (by TEST RMSE) =====")
if best["params"] is None:
    print("No successful run in the search space.")
else:
    print("Best TEST metrics:")
    print(f"  RMSE = {best['rmse_test']:.6f}")
    print(f"  MSE  = {best['mse_test']:.6f}")
    print(f"  MAE  = {best['mae_test']:.6f}")
    print(f"  R2   = {best['r2_test']:.6f}")
    print("\nBest params:")
    for k in keys:
        print(f"  {k} = {best['params'][k]}")
    print(f"  USE_REVIN = {best['params']['USE_REVIN']}")
    print(f"  USE_COVARIATES = {best['params']['USE_COVARIATES']}")
    print(f"  RANDOM_STATE = {best['params']['RANDOM_STATE']}")